# 🏋️ PRÁCTICA · Estructuras de control
## RA5 · Excepciones y código robusto

Cuaderno de **ejercicios de práctica** para prepararte el examen del RA5. Puedes repetir cada ejercicio las veces que quieras y comprobar tu solución ejecutando la celda de **comprobación** que sigue a cada uno.

**Niveles:** ⭐ sencillo · ⭐⭐ intermedio · ⭐⭐⭐ avanzado

| Nº | Criterios que practica | Nivel | Tema |
|---|---|---|---|
| 1 | a | ⭐ | Excepción o error de sintaxis |
| 2 | b, c | ⭐⭐ | `try/except/else/finally`: divisiones seguras |
| 3 | c | ⭐ | Capturar y tratar: lectura de enteros |
| 4 | d | ⭐⭐ | Excepciones propias: control de stock |
| 5 | d | ⭐⭐⭐ | Relanzar y encadenar: configuración JSON |
| 6 | e | ⭐⭐ | Depurar a partir del *traceback* |
| 7 | f | ⭐⭐⭐ | Código robusto con ficheros |

> **Consejo:** captura **excepciones concretas** (`ValueError`, `KeyError`...), nunca un `except:` vacío, y trata el error con una estrategia clara: valor por defecto, reintento, aviso o propagación.

In [20]:
# Celda de utilidades (NO modificar)
import hashlib
def _h(x):
    return hashlib.sha256(repr(x).encode()).hexdigest()[:8]
print("Listo")

Listo


---
## Ejercicio 1 · Criterio **a)** · ⭐

Clasifica cada fragmento: escribe `"sintaxis"` si Python lo rechaza **antes de ejecutarlo**, o el **nombre de la excepción** que se produce al ejecutarlo.

| Nº | Fragmento |
|---|---|
| 1 | `int("12.5")` |
| 2 | `print("a" + 1)` |
| 3 | `lista = [1, 2, 3` |
| 4 | `import modulo_que_no_existe` |
| 5 | `d = {}` ⏎ `d["x"]` |
| 6 | `for i in range(3)` ⏎ `    print(i)` *(falta algo en la primera línea)* |

In [25]:
clasificacion = {1: "ValueError", 2: "TypeError", 3: "sintaxis", 4: "ModuleNotFoundError", 5: "KeyError", 6: "sintaxis"}   # TU RESPUESTA AQUÍ

In [26]:
# Comprobación (NO modificar)
esperado = {1: "3200aae0", 2: "64efeab0", 3: "bb691598",
            4: "0d143304", 5: "adf8bb38", 6: "bb691598"}
for k, h in esperado.items():
    assert _h(clasificacion[k].strip()) == h, f"Revisa la respuesta {k}"
print("✔ Ejercicio 1 superado (puedes comprobarlas ejecutando cada fragmento en una celda)")

✔ Ejercicio 1 superado (puedes comprobarlas ejecutando cada fragmento en una celda)


---
## Ejercicio 2 · Criterios **b) y c)** · ⭐⭐

Escribe `dividir_pares(pares)`: recibe una lista de pares `(a, b)` y devuelve un diccionario:

```python
{"resultados": [...], "fallos": <nº de pares que fallaron>, "intentos": <nº total de pares>}
```

- `resultados` tiene un elemento por par: `a / b`, o `None` si la división falla (`ZeroDivisionError` o `TypeError`).
- Usa la estructura **`try / except / else / finally`**: guarda el resultado en el `else`, cuenta el fallo en el `except` y cuenta los intentos en el `finally`.

In [30]:
def dividir_pares(pares):

    resultados = []
    fallos = 0
    intentos = 0

    for a, b in pares:
        try:
            division = a / b
        except (ZeroDivisionError, TypeError):
            resultados.append(None)
            fallos += 1
        else:
            resultados.append(division)
        finally:
            intentos += 1

    return {
        "resultados": resultados,
        "fallos": fallos,
        "intentos": intentos
    }

In [31]:
# Comprobación (NO modificar)
assert dividir_pares([(10, 2), (1, 0), (6, "a"), (9, 3)]) == {"resultados": [5.0, None, None, 3.0], "fallos": 2, "intentos": 4}
assert dividir_pares([]) == {"resultados": [], "fallos": 0, "intentos": 0}
print("✔ Ejercicio 2 superado (el profesorado comprueba el uso de try/except/else/finally)")

✔ Ejercicio 2 superado (el profesorado comprueba el uso de try/except/else/finally)


---
## Ejercicio 3 · Criterio **c)** · ⭐

Escribe `leer_enteros(textos)` que devuelva la tupla `(enteros, rechazados)`: la lista de enteros que se han podido convertir (`int()`) y la lista de textos que **no** se pudieron convertir (por `ValueError` o `TypeError`), en el mismo orden en que aparecen.

In [18]:
def leer_enteros(textos):

    enteros = []
    rechazados = []

    for elemento in textos:
        try:
            numero = int(elemento)
            enteros.append(numero)
        except (ValueError, TypeError):
            rechazados.append(elemento)
    return enteros,rechazados

In [19]:
# Comprobación (NO modificar)
assert leer_enteros(["5", " 7 ", "3.5", None, "x", "-2"]) == ([5, 7, -2], ["3.5", None, "x"])
assert leer_enteros([]) == ([], [])
print("✔ Ejercicio 3 superado")

✔ Ejercicio 3 superado


---
## Ejercicio 4 · Criterio **d)** · ⭐⭐

Control de stock con **excepciones propias**:

1. Crea `ProductoDesconocidoError(Exception)` y `StockInsuficienteError(Exception)`. Esta última guarda en sus atributos `disponible` y `pedido`.
2. `retirar_stock(almacen, producto, cantidad)`: `almacen` es un diccionario `{producto: unidades}`. Si el producto no está, **lanza** `ProductoDesconocidoError`; si `cantidad` supera las unidades, **lanza** `StockInsuficienteError(disponible, pedido)`; en otro caso resta las unidades.
3. `procesar_pedidos(almacen, pedidos)`: recorre una lista de `(producto, cantidad)` y devuelve la lista de mensajes: `"ok"` si se retiró, `"desconocido"` si el producto no existe, o `"faltan N"` (con las unidades que faltan) si no hay stock suficiente.

In [8]:
class ProductoDesconocidoError(Exception):
    pass


class StockInsuficienteError(Exception):
    def __init__(self, disponible, pedido):
        self.disponible = disponible
        self.pedido = pedido
        super().__init__(f"Stock insuficiente: disponible {disponible}, pedido {pedido}")


def retirar_stock(almacen, producto, cantidad):
    if producto not in almacen:
        raise ProductoDesconocidoError(f"El producto '{producto}' no existe")

    disponible = almacen[producto]

    if cantidad > disponible:
        raise StockInsuficienteError(disponible, cantidad)

    almacen[producto] -= cantidad


def procesar_pedidos(almacen, pedidos):
    mensajes = []

    for producto, cantidad in pedidos:
        try:
            retirar_stock(almacen, producto, cantidad)
            mensajes.append("ok")
        except ProductoDesconocidoError:
            mensajes.append("desconocido")
        except StockInsuficienteError as e:
            faltan = e.pedido - e.disponible
            mensajes.append(f"faltan {faltan}")

    return mensajes


In [9]:
# Comprobación (NO modificar)
assert issubclass(ProductoDesconocidoError, Exception) and issubclass(StockInsuficienteError, Exception)
alm = {"pan": 10, "leche": 3}
try:
    retirar_stock(alm, "vino", 1)
    raise AssertionError("Debía lanzar ProductoDesconocidoError")
except ProductoDesconocidoError:
    pass
try:
    retirar_stock(alm, "leche", 5)
    raise AssertionError("Debía lanzar StockInsuficienteError")
except StockInsuficienteError as e:
    assert (e.disponible, e.pedido) == (3, 5)
retirar_stock(alm, "pan", 4)
assert alm == {"pan": 6, "leche": 3}
alm2 = {"pan": 10, "leche": 3}
assert procesar_pedidos(alm2, [("pan", 4), ("leche", 5), ("vino", 1), ("pan", 6)]) == ["ok", "faltan 2", "desconocido", "ok"]
assert alm2 == {"pan": 0, "leche": 3}
print("✔ Ejercicio 4 superado")

✔ Ejercicio 4 superado


---
## Ejercicio 5 · Criterio **d)** · ⭐⭐⭐

Crea `ConfigInvalidaError(Exception)` y la función `cargar_config(texto)` que reciba un texto JSON (usa `json.loads`) y devuelva el diccionario con `"host"` (texto) y `"puerto"` (entero). Debe:

- Si el JSON **no es válido** (`json.JSONDecodeError`): **lanzar** `ConfigInvalidaError` **encadenando** la excepción original con `raise ... from e`.
- Si falta `"host"` o `"puerto"`: lanzar `ConfigInvalidaError` con un mensaje que indique qué clave falta.
- Si `"puerto"` no es un entero entre 1 y 65535: lanzar `ConfigInvalidaError`.

In [ ]:
import json
class ConfigInvalidaError(Exception):
    pass

def cargar_config(texto):
    try:
        datos = json.loads(texto)
    except json.JSONDecodeError as e:
        raise ConfigInvalidaError("El texto no sigue el formato JSON")




In [ ]:
# Comprobación (NO modificar)
assert cargar_config('{"host": "localhost", "puerto": 8080}') == {"host": "localhost", "puerto": 8080}

try:
    cargar_config("{no es json")
    raise AssertionError("Debía lanzar ConfigInvalidaError")
except ConfigInvalidaError as e:
    assert isinstance(e.__cause__, json.JSONDecodeError), "Usa 'raise ... from e' para encadenar la causa"

for texto, clave in [('{"puerto": 80}', "host"), ('{"host": "a"}', "puerto")]:
    try:
        cargar_config(texto)
        raise AssertionError("Debía lanzar ConfigInvalidaError")
    except ConfigInvalidaError as e:
        assert clave in str(e), f"El mensaje debe mencionar la clave que falta ({clave})"

for malo in ('{"host": "a", "puerto": "80"}', '{"host": "a", "puerto": 0}', '{"host": "a", "puerto": 70000}'):
    try:
        cargar_config(malo)
        raise AssertionError(f"Debía rechazar {malo}")
    except ConfigInvalidaError:
        pass
print("✔ Ejercicio 5 superado")

---
## Ejercicio 6 · Criterio **e)** · ⭐⭐

Se te da esta función, que contiene varios fallos:

```python
def precio_total(carrito, precios):
    total = 0
    for producto, cantidad in carrito.items():
        total += precios[producto] * cantidad
    return total / len(carrito)
```

**6.1** Sin ejecutarlas (o ejecutándolas después para confirmarlo), indica la **excepción** que lanza cada llamada (o `"ninguna"`):

| Clave | Llamada |
|---|---|
| `"a"` | `precio_total({"pan": 2}, {"pan": 1.5})` |
| `"b"` | `precio_total({"leche": 1}, {"pan": 1.5})` |
| `"c"` | `precio_total({"pan": "2"}, {"pan": 1.5})` |
| `"d"` | `precio_total({}, {})` |

**6.2** Copia en `linea_del_fallo_b` la **línea de código** en la que se produce la excepción de la llamada `b`.

In [ ]:
def precio_total(carrito, precios):
    total = 0
    for producto, cantidad in carrito.items():
        total += precios[producto] * cantidad
    return total / len(carrito)

precio_total({"pan": 2}, {"pan": 1.5})
#precio_total({"leche": 1}, {"pan": 1.5})
#precio_total({"pan": "2"}, {"pan": 1.5})

precio_total({}, {})

In [16]:
excepciones = {"a": "ninguna", "b": "KeyError", "c": "TypeError", "d": "ZeroDivisionError"}   # TU RESPUESTA AQUÍ
linea_del_fallo_b = "total += precios[producto] * cantidad"

In [17]:
# Comprobación (NO modificar)
assert _h(sorted(excepciones.items())) == "79f5fd62", "Revisa las excepciones"
assert "precios[producto]" in linea_del_fallo_b, "Revisa la línea del fallo"
print("✔ Ejercicio 6 superado (compruébalo ejecutando la función con cada llamada dentro de un try/except)")

✔ Ejercicio 6 superado (compruébalo ejecutando la función con cada llamada dentro de un try/except)


---
## Ejercicio 7 · Criterio **f)** · ⭐⭐⭐

Escribe `cargar_precios(ruta)` como **código robusto**. El fichero tiene líneas con el formato `producto;precio`. Debe devolver un diccionario `{producto: precio_float}` y:

- Abrir el fichero con **`with`**.
- **Ignorar** las líneas vacías, sin `;` o con un precio que no sea un número.
- Si el producto aparece varias veces, **vale la última** línea válida.
- Si el fichero **no existe**, devolver un diccionario vacío `{}` (sin lanzar la excepción).

In [ ]:
# Preparación (NO modificar)
with open("precios_ok.txt", "w", encoding="utf-8") as f:
    f.write("pan;1.20\nleche;0.95\n")
with open("precios_mal.txt", "w", encoding="utf-8") as f:
    f.write("pan;1.20\n\nleche\nqueso;abc\nvino;8.5\npan;1.35\n;;\n")

def cargar_precios(ruta):



In [ ]:
# Comprobación (NO modificar)
assert cargar_precios("precios_ok.txt") == {"pan": 1.2, "leche": 0.95}
assert cargar_precios("precios_mal.txt") == {"pan": 1.35, "vino": 8.5}, "Debe ignorar líneas mal formadas y quedarse con el último valor de cada producto"
assert cargar_precios("no_existe.txt") == {}
print("✔ Ejercicio 7 superado (el profesorado comprueba el uso de with y la gestión de errores)")

---
## ✅ Cuando termines

Vuelve a la tabla del principio: repite los ejercicios de los criterios que más te cuesten. Cuando todos te salgan a la primera, estás listo para el examen del RA5.